<a href="https://colab.research.google.com/github/SANGHATI23/neuroimaging-fhir-omop-fidelity/blob/main/ICHI2027_Colab_07_Pinned_MICDM_Schema_and_Vocabulary.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pathlib import Path
import os, re, sys, io, csv, json, hashlib, shutil, subprocess, sqlite3, gzip
from datetime import datetime, timezone

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except ImportError:
    pass

BASE = Path('/content')
PROJECT = BASE / 'drive/MyDrive/NeuroFHIR_ICHI2027'
PHASE06_DIR = PROJECT / 'phase06_outputs'
OUT_DIR = PROJECT / 'phase07_outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)
WORK = BASE / 'ICHI2027_Phase07_private_work'
WORK.mkdir(exist_ok=True, parents=True)
OUT_JSON = OUT_DIR / 'ICHI2027_Phase07_SHAREABLE.json'
SQL_PATH = OUT_DIR / 'ICHI2027_Phase07_DERIVED_Staging_Schema.sql'
CONTRACT_PATH = OUT_DIR / 'ICHI2027_Phase07_Public_Field_Contract.json'
DB_PATH = OUT_DIR / 'ICHI2027_Phase07_EMPTY_Staging_Schema.sqlite'

def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def sh(cmd, cwd=None):
    p = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if p.returncode:
        raise RuntimeError(f'Command {cmd[0]} failed with code {p.returncode}: '+p.stderr[-800:])
    return p.stdout.strip()

NOW_UTC = datetime.now(timezone.utc).isoformat()


Mounted at /content/drive


In [2]:
source_candidates = [
    PHASE06_DIR / 'ICHI2027_Phase06_SHAREABLE.json',
    BASE / 'ICHI2027_Phase06_SHAREABLE.json',
]
phase06_file = next((p for p in source_candidates if p.is_file()), None)
if phase06_file is None:
    from google.colab import files
    uploaded = files.upload()
    exact = 'ICHI2027_Phase06_SHAREABLE.json'
    if exact not in uploaded:
        raise FileNotFoundError(f'Required: {exact}')
    phase06_file = WORK / exact
    phase06_file.write_bytes(uploaded[exact])
p06 = json.loads(phase06_file.read_text(encoding='utf-8'))
assert p06.get('phase') == '06_original_tfl_replay_and_source_provenance'
assert p06['pinned_repositories']['imagewg']['verified'] is True
assert p06['pinned_repositories']['ohdsi_tfl']['verified'] is True
p06_sha = sha256_file(phase06_file)


In [3]:
IMAGEWG_URL = 'https://github.com/OHDSI/ImageWG.git'
IMAGEWG_SHA = p06['pinned_repositories']['imagewg']['actual_commit']
imagewg = WORK / 'ImageWG'
if not (imagewg / '.git').exists():
    sh(['git', 'clone', '--quiet', IMAGEWG_URL, str(imagewg)])
sh(['git', 'checkout', '--quiet', '--detach', IMAGEWG_SHA], cwd=imagewg)
assert sh(['git','rev-parse','HEAD'], cwd=imagewg) == IMAGEWG_SHA

spec_path = imagewg / 'inst/csv/new_csv/OMOP_CDM_Medical_Imaging_Ex_Field_Level.csv'
assert spec_path.is_file(), f'Pinned field-spec missing: {spec_path}'
actual_spec_sha = sha256_file(spec_path)
assert actual_spec_sha == p06['micdm']['field_spec_sha256'], 'Pinned MI-CDM field-spec checksum changed'


In [4]:
import pandas as pd

spec_raw = pd.read_csv(spec_path, dtype=str, keep_default_na=False, encoding='utf-8-sig')
spec_raw.columns = [re.sub(r'\s+', '', x.replace('\u00a0',' ')).strip() for x in spec_raw.columns]
mandatory_headers = {'cdmTableName','cdmFieldName','cdmDatatype','isRequired'}
if missing_headers := sorted(mandatory_headers - set(spec_raw.columns)):
    raise ValueError(f'Unknown source-spec column layout: {missing_headers}; observed={list(spec_raw.columns)}')

def bool_flag(v):
    s = str(v).replace('\u00a0',' ').strip().lower()
    if s in ('', 'n','no','false','0','optional'): return False
    if s in ('y','yes','true','1','required'): return True
    raise ValueError(f'Unrecognized required/PK flag: {v!r}')

def source_val(row, name):
    return str(row.get(name, '')).replace('\u00a0',' ').strip()

TABLES = ('IMAGE_OCCURRENCE', 'IMAGE_FEATURE')
rows = []
for _, raw in spec_raw.iterrows():
    tab = source_val(raw, 'cdmTableName').upper()
    if tab not in TABLES: continue
    name = source_val(raw, 'cdmFieldName')
    dtype = source_val(raw, 'cdmDatatype')
    if not re.fullmatch(r'[A-Za-z_][A-Za-z_0-9]*', name):
        raise ValueError(f'Unsafe/unknown source field name: {name!r}')
    if not dtype:
        raise ValueError(f'Missing source datatype: {tab}.{name}')
    rows.append({
        'table': tab,
        'field': name,
        'source_datatype': dtype,
        'required': bool_flag(source_val(raw,'isRequired')),
        'pk_flag_in_spec': bool_flag(source_val(raw,'isPrimaryKey')),
        'fk_flag_in_spec': bool_flag(source_val(raw,'isForeignKey')),
        'fk_table_in_spec': source_val(raw,'fkTableName'),
        'fk_column_in_spec': source_val(raw,'fkFrieldName') or source_val(raw,'fkFieldName'),
    })

if not rows: raise ValueError('No IMAGE_OCCURRENCE or IMAGE_FEATURE fields found in pinned source spec')
for table in TABLES:
    entries = [r for r in rows if r['table'] == table]
    if not entries: raise ValueError(f'Pinned spec has no {table} fields')
    fields = [r['field'].lower() for r in entries]
    if len(fields) != len(set(fields)):
        raise ValueError(f'Duplicate column names found in pinned {table} specification')

spec_summary = {
    tab: {
        'field_count': len([r for r in rows if r['table'] == tab]),
        'required_fields': [r['field'] for r in rows if r['table'] == tab and r['required']],
        'source_documented_pk': [r['field'] for r in rows if r['table'] == tab and r['pk_flag_in_spec']],
    } for tab in TABLES
}


In [5]:
# This is a transparent local SQLite derivative of a pinned field-level CSV, not official ImageWG DDL.
def sqlite_type(source_datatype):
    s = source_datatype.strip().lower().replace(' ', '')
    if s in ('integer','int','bigint','smallint','tinyint'): return 'INTEGER'
    if s in ('real','float','double','decimal','number') or re.fullmatch(r'(decimal|numeric)\([0-9]+,[0-9]+\)',s):
        return 'REAL'
    if s in ('text','string','date','datetime','timestamp','time','boolean','bool'):
        return 'TEXT' if s not in ('boolean','bool') else 'INTEGER'
    if re.fullmatch(r'(n?varchar|char|nchar)\((max|[0-9]+)\)',s): return 'TEXT'
    raise ValueError('Unknown MI-CDM datatype; do not guess a target type: '+source_datatype)

def quote_id(s):
    if not re.fullmatch(r'[A-Za-z_][A-Za-z_0-9]*',s): raise ValueError(s)
    return '"'+s+'"'

def generate_sql(fields):
    ddls = []
    external_fks, unspecified_fks = [], []
    for tab in TABLES:
        rtab = [r for r in fields if r['table']==tab]
        cols=[]; internal_constraints=[]
        pk = [r['field'] for r in rtab if r['pk_flag_in_spec']]
        if len(pk)>1:
            raise ValueError('Composite key not automatically supported; review pinned spec: '+tab)
        for r in rtab:
            typename=sqlite_type(r['source_datatype'])
            c=quote_id(r['field'])+' '+typename
            if r['required']: c+=' NOT NULL'
            if r['field'] in pk: c+=' PRIMARY KEY'
            match = re.fullmatch(r'(?:n?varchar|char|nchar)\(([0-9]+)\)', r['source_datatype'].strip().lower().replace(' ',''))
            if match: c+=' CHECK(length('+quote_id(r['field'])+') <= '+str(int(match.group(1)))+')'
            cols.append('  '+c)
            if r['fk_flag_in_spec']:
                target = r['fk_table_in_spec'].upper()
                targetcol = r['fk_column_in_spec']
                target_fields={e['field'].lower() for e in fields if e['table']==target}
                if target in TABLES and targetcol.lower() in target_fields:
                    internal_constraints.append('  FOREIGN KEY ('+quote_id(r['field'])+') REFERENCES '+quote_id(target)+' ('+quote_id(targetcol)+')')
                elif target and target not in TABLES:
                    external_fks.append({'table':tab,'field':r['field'],'target':target,'target_field':targetcol})
                else:
                    unspecified_fks.append({'table':tab,'field':r['field'],'fk_table_source':target,'fk_field_source':targetcol})
        ddls.append('CREATE TABLE '+quote_id(tab)+' (\n'+',\n'.join(cols+internal_constraints)+'\n);')
    return 'PRAGMA foreign_keys=ON;\n\n'+'\n\n'.join(ddls)+'\n', external_fks, unspecified_fks

sql, external_fks, unspecified_fks = generate_sql(rows)
SQL_PATH.write_text(sql, encoding='utf-8')
contract={
    'status':'DERIVED_FROM_PINNED_IMAGEWG_FIELD_CSV_NOT_OFFICIAL_DDL',
    'imagewg_commit':IMAGEWG_SHA,
    'source_field_spec_sha256':actual_spec_sha,
    'rows': rows,
    'table_summary':spec_summary,
    'external_fk_definitions_not_enforced_in_local_staging':external_fks,
    'fk_definitions_requiring_manual_review':unspecified_fks,
    'generated_sql_sha256':sha256_file(SQL_PATH),
}
CONTRACT_PATH.write_text(json.dumps(contract, indent=2), encoding='utf-8')


9287

In [6]:
# A zero-record structural check; no patient records or invented values are inserted.
if DB_PATH.exists(): DB_PATH.unlink()
con=sqlite3.connect(DB_PATH)
try:
    con.executescript(sql)
    sql_introspection={}
    for table in TABLES:
        names=[r[1] for r in con.execute('PRAGMA table_info('+table+')')]
        expected=[r['field'] for r in rows if r['table']==table]
        assert names == expected, f'Staging schema mismatch: {table}'
        assert con.execute('SELECT COUNT(*) FROM '+table).fetchone()[0] == 0
        sql_introspection[table]={
            'fields_verified':len(names),
            'schema_column_names_equal_pinned_spec':True,
            'private_or_synthetic_rows_inserted':0,
            'fk_constraint_count':len(con.execute('PRAGMA foreign_key_list('+table+')').fetchall()),
        }
    con.commit()
finally:
    con.close()

sqlite_schema_sha=sha256_file(DB_PATH)


In [7]:
# Optional local Athena CONCEPT.csv snapshot: no vocabulary IDs are guessed or assigned.
import gzip
VOCAB_DIR=PROJECT/'vocabulary'
vocab_candidates=[
    VOCAB_DIR/'CONCEPT.csv', VOCAB_DIR/'CONCEPT.csv.gz',
    VOCAB_DIR/'CONCEPT.tsv', VOCAB_DIR/'CONCEPT.tsv.gz',
    BASE/'CONCEPT.csv', BASE/'CONCEPT.csv.gz',
]
vocab_path=next((p for p in vocab_candidates if p.is_file()),None)
concept_evidence={'snapshot_found':bool(vocab_path),'status':'NO_LOCAL_SNAPSHOT','candidate_concepts_are_not_approved_mappings':True}

if vocab_path:
    opener=gzip.open if vocab_path.name.endswith('.gz') else open
    with opener(vocab_path, 'rt',encoding='utf-8-sig',errors='replace',newline='') as f:
        header=f.readline()
    delim='\t' if header.count('\t')>header.count(',') else ','
    columns=[x.strip().lower() for x in next(csv.reader([header],delimiter=delim))]
    required_cols={'concept_id','concept_name','vocabulary_id','domain_id','standard_concept','invalid_reason'}
    if missing:=sorted(required_cols-set(columns)):
        concept_evidence.update({'status':'INVALID_COLUMNS','missing_columns':missing})
    else:
        terms={
            'mri':['magnetic resonance','mri'],
            'hippocampus':['hippocamp'],
            'entorhinal':['entorhin'],
            'middle_temporal':['middle temporal'],
            'ventricles':['ventricl'],
            'intracranial_volume':['intracranial volume'],
        }
        seen={k:0 for k in terms}
        sample={k:[] for k in terms}
        row_count=0
        # Small bounded candidate lists; no autoselection of OMOP concepts.
        use_cols=[c for c in columns if c in required_cols|{'valid_start_date','valid_end_date'}]
        for chunk in pd.read_csv(vocab_path, compression='infer',sep=delim,dtype=str,keep_default_na=False,
                                 usecols=use_cols,chunksize=100000,encoding='utf-8-sig'):
            row_count+=len(chunk)
            active=chunk[(chunk['standard_concept'].str.upper()=='S') &
                         (chunk['invalid_reason'].str.strip()=='')]
            lower=active['concept_name'].str.lower()
            for key,substrings in terms.items():
                mask=False
                for term in substrings:
                    termmask=lower.str.contains(term,regex=False,na=False)
                    mask=mask|termmask
                hits=active[mask]
                seen[key]+=len(hits)
                if len(sample[key])<15:
                    needed=15-len(sample[key])
                    sample[key].extend(hits[['concept_id','concept_name','vocabulary_id','domain_id']].head(needed).to_dict('records'))
        concept_evidence={
            'snapshot_found':True,'status':'LOCAL_SNAPSHOT_SCANNED_NOT_MAPPED',
            'snapshot_sha256':sha256_file(vocab_path),
            'rows_scanned':row_count,'delimiter': 'TAB' if delim=='\t' else 'COMMA',
            'standard_noninvalid_candidate_counts':seen,
            'bounded_candidate_lists_for_review_only':sample,
            'candidate_concepts_are_not_approved_mappings':True,
        }


In [8]:
# Old Phase06 unit flag was a lexical match to encoded webpage data, not a proven field/unit crosswalk.
old_doc_evidence=p06.get('official_adni_public_document_evidence',[])
base64_like = []
for doc in old_doc_evidence:
    for item in doc.get('matches',[]):
        snip=item.get('snippet','')
        if len(re.findall(r'[A-Za-z0-9+/]{75,}={0,2}',snip)):
            base64_like.append(item.get('term'))
unit_status={
    'phase06_claim_reexamined':'NOT_VALID_FOR_EXACT_LOCAL_EXPORT',
    'prior_snippet_contains_base64_like_text':bool(base64_like),
    'exact_local_export_units_approved':False,
    'reason':'A generic webpage lexical match cannot prove units for the uploaded local MRI export.',
}

# This folder is optional; inventory names/checksums only, not confidential file contents.
provenance_dir=PROJECT/'source_provenance'
allowed_suffixes={'.pdf','.csv','.xlsx','.xls','.json','.txt','.md','.zip'}
provenance_files=[]
if provenance_dir.exists():
    for p in sorted(provenance_dir.rglob('*')):
        if p.is_file() and p.suffix.lower() in allowed_suffixes:
            provenance_files.append({'extension':p.suffix.lower(),'sha256':sha256_file(p),'size_bytes':p.stat().st_size})
provenance_status={
    'optional_source_provenance_files_detected':len(provenance_files),
    'file_hashes_without_names_or_contents':provenance_files,
    'source_export_specific_unit_verification':'NOT_PERFORMED',
    'dicom_study_uid_verification':'NOT_PERFORMED',
    'dicom_series_uid_verification':'NOT_PERFORMED',
}


In [9]:
# Phase06 runner completion is not evidence that its output is an MRI/FHIR comparator adapter.
tfl_replay={
    'phase06_frozen_runner_execution_completed':p06['tfl_v15_runner']['execution'].get('completed',False),
    'phase06_runner_sha256':p06['tfl_v15_runner']['sha256'],
    'MRI_specific_TFL_adapter_executed':False,
    'frozen_v15_replayed_on_current_real_MRI_FHIR_data':False,
    'comparative_ETL_scores_available':False,
}

phase07={
    'project':'ICHI2027_neuroimaging_fhir_omop_fidelity',
    'phase':'07_pinned_micdm_derived_schema_and_vocabulary_preflight',
    'run_utc':NOW_UTC,
    'results_status':'SCHEMA_AND_VOCABULARY_EVIDENCE_ONLY_NO_MRI_OMOP_LOAD_NO_COMPARISON',
    'privacy':'Shareable report excludes ADNI participant rows and identifiers; local SQLite tables contain zero rows.',
    'phase06_json_sha256':p06_sha,
    'frozen_imagewg_revision':IMAGEWG_SHA,
    'micdm_field_spec_sha256':actual_spec_sha,
    'micdm_contract':spec_summary,
    'pinned_image_feature_numeric_value_column_present':any(
        r['table']=='IMAGE_FEATURE' and r['field'].lower() in ('value_as_number','value','numeric_value') for r in rows
    ),
    'derived_empty_sqlite_staging':{
        'status':'LOCALLY_GENERATED_FROM_PINNED_FIELD_SPEC_NOT_OFFICIAL_DDL',
        'sql_sha256':sha256_file(SQL_PATH),
        'sqlite_sha256':sqlite_schema_sha,
        'schema_introspection':sql_introspection,
        'external_fk_definitions_not_enforced_in_staging':external_fks,
        'fks_requiring_review':unspecified_fks,
    },
    'omop_vocabulary':concept_evidence,
    'unit_evidence_correction':unit_status,
    'optional_source_provenance_inventory':provenance_status,
    'original_tfl':tfl_replay,
    'gates':{
        'pinned_imagewg_csv_checksum_verified':True,
        'derived_empty_staging_schema_built_and_introspected':True,
        'official_micdm_DDL_verified':False,
        'omop_vocabulary_snapshot_present_and_scanned':concept_evidence['status']=='LOCAL_SNAPSHOT_SCANNED_NOT_MAPPED',
        'required_omop_concept_mappings_approved':False,
        'exact_export_measurement_unit_authoritatively_verified':False,
        'verified_DICOM_study_and_series_UIDs':False,
        'source_procedure_occurrence_mapping_approved':False,
        'source_to_target_numeric_measurement_mapping_approved':False,
        'real_micdm_data_load_done':False,
        'genuine_four_arm_experiment_done':False,
    },
    'four_arm_ETL_scores':None,
    'source_vs_target_cohort_statistics':None,
    'next_stage_gate':'VALIDATE_EXACT_ADNI_SOURCE_DICTIONARY_AND_OMOP_VOCABULARY_THEN_BUILD_AUTHENTIC_MI_CDM_ADAPTER',
}
OUT_JSON.write_text(json.dumps(phase07,indent=2,ensure_ascii=False),encoding='utf-8')
print(json.dumps({
    'output':str(OUT_JSON),
    'staging_schema_tables':{k:v['fields_verified'] for k,v in sql_introspection.items()},
    'vocabulary_status':concept_evidence['status'],
    'unit_status':'UNVERIFIED',
    'comparative_results':None,
},indent=2))


{
  "output": "/content/drive/MyDrive/NeuroFHIR_ICHI2027/phase07_outputs/ICHI2027_Phase07_SHAREABLE.json",
  "staging_schema_tables": {
    "IMAGE_OCCURRENCE": 11,
    "IMAGE_FEATURE": 12
  },
  "vocabulary_status": "NO_LOCAL_SNAPSHOT",
  "unit_status": "UNVERIFIED",
  "comparative_results": null
}


In [10]:
try:
    from google.colab import files
    files.download(str(OUT_JSON))
except ImportError:
    print(OUT_JSON)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>